# Stateful Streaming in Spark Structured Streaming

## Why State Matters in Streaming

Stateless streaming processes each micro-batch independently — every record is handled without memory of what came before. That works for simple transformations (filters, maps) but breaks down the moment you need to *aggregate across time*.

**Stateful streaming** maintains per-key, per-window counters, sums, and arbitrary state objects across micro-batches. Spark stores this state in an in-memory hash map that is backed by a write-ahead log on durable storage.

### What Spark stores per key per partition

| Layer | What is stored | Where |
|-------|---------------|-------|
| In-memory | Current aggregate value for each (key, window) pair | Executor JVM heap |
| State store | RocksDB or HDFS-backed delta files, one per partition | Checkpoint directory |
| WAL (write-ahead log) | Sequence of state updates before they are applied | Checkpoint directory |

Each partition independently manages its slice of the state. Because keys are hashed onto partitions by `hashPartitioner`, all events for a given key always route to the same executor — which is what makes incremental aggregation possible without shuffling the state itself.

**Key take-away:** state is not free. For high-cardinality keys over long windows, state can grow until the executor runs out of memory. The watermark mechanism (covered later in this notebook) is the primary tool for bounding state growth.

## Event-Time vs Processing-Time

```
  Wall clock (processing time)
  ─────────────────────────────────────────────────────────►
  t=0         t=30s        t=60s        t=90s        t=120s
  │           │            │            │            │
  │  [e1,t=0] [e2,t=25s]   │ [e4,t=55s] │  [e5,t=50s]│   ← events arrive at Spark
  │           │            │  [e3,t=40s]│            │
  └───────────┴────────────┴────────────┴────────────┘

  Event time (embedded in the record)
  ─────────────────────────────────────────────────────────►
  t=0     t=25s   t=40s   t=50s   t=55s
  e1      e2      e3      e5(LATE) e4
```

Notice **e5**: its event-time is `t=50s` but it arrives at Spark at `t=90s`. If you group by processing-time, e5 falls in the wrong window. If you group by **event-time**, e5 correctly lands in the `[40s, 60s)` window — *as long as it arrives before the watermark expires that window*.

### Why event-time windowing is correct for real workloads

- **Mobile apps** buffer events offline; they can arrive minutes or hours late.
- **IoT sensors** have network jitter of tens of seconds.
- **Kafka consumer lag** can shift arrival times by minutes under heavy load.

Processing-time windows would miscount all of these. Event-time windows are immune because the timestamp is *in the data*, not in Spark's system clock.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType
import time

spark = (
    SparkSession.builder
    .appName("AdvancedStatefulStreaming")
    .master("spark://spark-master:7077")
    # How often Spark checks for new data (micro-batch interval).
    # For this demo we keep it short so results appear quickly.
    .config("spark.sql.streaming.microBatchDuration", "5s")
    # State store provider — HDFSBackedStateStoreProvider persists to the
    # checkpoint directory; RocksDB is available since Spark 3.2 for lower
    # memory pressure on large state.
    .config(
        "spark.sql.streaming.stateStore.providerClass",
        "org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider"
    )
    # Shuffle partitions: keep low for local demo to avoid 200 tiny tasks
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

In [ ]:
# ---------------------------------------------------------------------------
# Rate source — Spark's built-in synthetic stream
# It emits (timestamp, value) rows at a configurable rate.
# We use it to simulate a real event stream without needing Kafka.
# ---------------------------------------------------------------------------

# 10 rows per second, starting immediately
raw_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)
    .load()
)

# The 'timestamp' column from the rate source IS the event-time.
# We add a synthetic 'key' column by bucketing the monotonically increasing
# 'value' into one of four groups — this simulates four event producers.
events = (
    raw_stream
    # Rename the built-in timestamp column to eventTime for clarity
    .withColumnRenamed("timestamp", "eventTime")
    # Create a low-cardinality key: 'sensor_A' … 'sensor_D'
    .withColumn(
        "sensorId",
        F.concat(
            F.lit("sensor_"),
            F.element_at(
                F.array(F.lit("A"), F.lit("B"), F.lit("C"), F.lit("D")),
                # Map value mod 4 → index 1–4 (element_at is 1-based)
                (F.col("value") % 4 + 1).cast("int")
            )
        )
    )
    # Simulate a random temperature reading between 20 and 40
    .withColumn(
        "temperature",
        (F.rand() * 20 + 20).cast("double")
    )
)

print("Stream schema:")
events.printSchema()

In [ ]:
# ---------------------------------------------------------------------------
# TUMBLING WINDOW (non-overlapping, fixed-size buckets)
# window("eventTime", "1 minute") creates a new bucket every 60 seconds.
# Each event belongs to EXACTLY ONE bucket.
# ---------------------------------------------------------------------------

tumbling_agg = (
    events
    .groupBy(
        # window() returns a struct column with .start and .end fields
        F.window("eventTime", "1 minute"),  # bucket width
        "sensorId"
    )
    .agg(
        F.count("*").alias("eventCount"),
        F.round(F.avg("temperature"), 2).alias("avgTemp"),
        F.round(F.max("temperature"), 2).alias("maxTemp")
    )
    # Flatten the window struct so the output columns are easier to read
    .withColumn("windowStart", F.col("window.start"))
    .withColumn("windowEnd",   F.col("window.end"))
    .drop("window")
    .orderBy("windowStart", "sensorId")
)

print("Tumbling window output schema:")
tumbling_agg.printSchema()

In [ ]:
# ---------------------------------------------------------------------------
# SLIDING WINDOW (overlapping)
# window("eventTime", windowDuration, slideDuration)
#   windowDuration = how long each bucket spans
#   slideDuration  = how often a new bucket starts
# Here: 2-minute window, sliding every 30 seconds.
# Each event therefore belongs to up to 4 buckets.
#
# Use case: rolling averages (e.g. "last 2 minutes" dashboard metrics)
# Cost:     state is proportionally larger — factor of (2min / 30s) = 4×
# ---------------------------------------------------------------------------

sliding_agg = (
    events
    .groupBy(
        F.window(
            "eventTime",
            "2 minutes",   # window duration
            "30 seconds"   # slide duration — a new window opens every 30s
        ),
        "sensorId"
    )
    .agg(
        F.count("*").alias("eventCount"),
        F.round(F.avg("temperature"), 2).alias("avgTemp")
    )
    .withColumn("windowStart", F.col("window.start"))
    .withColumn("windowEnd",   F.col("window.end"))
    .drop("window")
    .orderBy("windowStart", "sensorId")
)

print("Sliding window output schema:")
sliding_agg.printSchema()

In [ ]:
# ---------------------------------------------------------------------------
# WATERMARK — bounding state growth
#
# Without a watermark, Spark must keep state for EVERY window ever opened
# because a late event could theoretically arrive at any future micro-batch.
# That leaks memory indefinitely.
#
# .withWatermark("eventTime", "30 seconds") tells Spark:
#   "I guarantee events will not arrive more than 30 seconds late."
#   → Any window whose END is older than (max_seen_event_time - 30s) is safe
#     to finalize and evict from state.
#
# Rule of thumb: set the watermark to 2–3× your typical network/queue lag.
# ---------------------------------------------------------------------------

watermarked_agg = (
    events
    # Watermark MUST be applied before groupBy — order matters!
    .withWatermark("eventTime", "30 seconds")
    .groupBy(
        F.window("eventTime", "1 minute"),
        "sensorId"
    )
    .agg(
        F.count("*").alias("eventCount"),
        F.round(F.avg("temperature"), 2).alias("avgTemp")
    )
    .withColumn("windowStart", F.col("window.start"))
    .withColumn("windowEnd",   F.col("window.end"))
    .drop("window")
)

print("Watermarked aggregation plan (notice EventTimeWatermark node):")
watermarked_agg.explain()

In [ ]:
# ---------------------------------------------------------------------------
# MEMORY SINK — write results to an in-driver SQL table for easy inspection.
# NEVER use the memory sink in production; it doesn't scale and loses data
# on driver restart. Use it only for development/testing.
#
# outputMode:
#   "complete" — rewrite the entire result table every micro-batch (OK for
#                small aggregations; expensive for large state)
#   "update"   — only emit rows that changed since last batch
#   "append"   — only emit new rows (requires watermark + final windows)
# ---------------------------------------------------------------------------

query = (
    watermarked_agg
    .writeStream
    .outputMode("update")   # emit only changed aggregates each micro-batch
    .format("memory")       # write to an in-memory SQL table
    .queryName("sensor_window_agg")  # table name for spark.sql()
    .trigger(processingTime="5 seconds")  # run a micro-batch every 5s
    .start()
)

print(f"Query started: {query.name}  (id={query.id})")
print("Running for 30 seconds — collecting state …")

# Block for 30 seconds then stop automatically
query.awaitTermination(30_000)

print("\n=== Accumulated results (last snapshot) ===")
spark.sql("""
    SELECT sensorId, windowStart, windowEnd, eventCount, avgTemp
    FROM   sensor_window_agg
    ORDER  BY windowStart, sensorId
""").show(40, truncate=False)

## Checkpointing — Fault Tolerance for Stateful Streams

Checkpointing saves the complete state of a streaming query to durable storage so that it can be restarted from where it left off after a failure.

### What gets saved

| Checkpoint artefact | Contents | Written when |
|---------------------|----------|--------------|
| `offsets/` | Kafka offsets / rate-source sequence numbers processed so far | Before each micro-batch starts |
| `commits/` | Confirmation that a micro-batch completed successfully | After the sink write succeeds |
| `state/` | Full operator state (window aggregates, flatMapGroupsWithState, etc.) | After each micro-batch |
| `metadata` | Query metadata (query ID, Spark version) | On first start |

Spark uses a **two-phase commit** approach: it writes the offset before processing and the commit after. On restart, if there is an offset without a matching commit, Spark replays that micro-batch — ensuring **exactly-once** semantics end-to-end (assuming an idempotent sink).

### Why checkpointing is required in production

Without checkpointing:
- A driver crash loses all in-memory state (all window counts reset to zero).
- The query restarts from the *latest* offset — all events that arrived during downtime are skipped.
- You cannot change the query plan between restarts (Spark validates the saved plan against the new one).

### Code pattern

```python
query = (
    watermarked_agg
    .writeStream
    .outputMode("update")
    .format("parquet")
    .option("path", "/home/jovyan/data/output/sensor_agg")
    # Point to a directory on HDFS, S3, GCS, or ADLS.
    # Local filesystem works for single-node dev but not for HA clusters.
    .option("checkpointLocation", "/home/jovyan/data/checkpoints/sensor_agg")
    .trigger(processingTime="30 seconds")
    .start()
)
```

> **Production checklist**
> 1. Always set `checkpointLocation` to a path on distributed storage (HDFS/S3).
> 2. Use a unique checkpoint path per query — sharing corrupts both queries.
> 3. After changing the query plan significantly, delete the checkpoint and restart from the beginning (or use the `cleanSource` option carefully).

In [ ]:
# Gracefully stop any still-running streaming queries before stopping the session.
# This flushes pending micro-batches and releases executor resources cleanly.
for active_query in spark.streams.active:
    print(f"Stopping query: {active_query.name}")
    active_query.stop()

spark.stop()
print("SparkSession stopped.")